In [1]:
%%sql
WITH latest_quarter AS (
    SELECT year, quarter
    FROM dim_date
    ORDER BY year DESC, quarter DESC
    LIMIT 1
),
product_revenue AS (
    SELECT
        s.region,
        p.product_title,
        SUM(f.revenue) AS total_revenue
    FROM fact_sales f
    JOIN dim_product p
        ON f.product_id = p.product_id
    JOIN dim_store s
        ON f.store_id = s.store_id
    JOIN dim_date d
        ON f.date_key = d.date_key
    WHERE (d.year, d.quarter) IN (
        SELECT year, quarter
        FROM latest_quarter
    )
      AND f.revenue IS NOT NULL
    GROUP BY s.region, p.product_title
),
ranked_products AS (
    SELECT
        region,
        product_title,
        total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY region
            ORDER BY total_revenue DESC
        ) AS rn
    FROM product_revenue
)
SELECT
    region,
    product_title,
    total_revenue
FROM ranked_products
WHERE rn <= 5
ORDER BY region, total_revenue DESC;

StatementMeta(, 4a62ac9d-16fb-413d-8a0f-b8623e82b568, 4, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 3 fields>

In [ ]:
%%sql
WITH max_date AS (
    SELECT MAX(full_date) AS max_full_date
    FROM dim_date
),
customer_revenue AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(f.revenue) AS total_revenue_12m
    FROM fact_sales f
    JOIN dim_customer c
        ON f.customer_id = c.customer_id
    JOIN dim_date d
        ON f.date_key = d.date_key
    CROSS JOIN max_date m
    WHERE d.full_date >= ADD_MONTHS(m.max_full_date, -12)
      AND d.full_date <= m.max_full_date
      AND f.revenue IS NOT NULL
      AND f.order_status IN ('completed', 'shipped') 
    GROUP BY c.customer_id, c.customer_name
),
ranked_customers AS (
    SELECT
        customer_id,
        customer_name,
        total_revenue_12m,
        NTILE(3) OVER (ORDER BY total_revenue_12m DESC) AS value_group_num
    FROM customer_revenue
)
SELECT
    customer_id,
    customer_name,
    total_revenue_12m,
    CASE
        WHEN value_group_num = 1 THEN 'High Value'
        WHEN value_group_num = 2 THEN 'Mid Value'
        WHEN value_group_num = 3 THEN 'Low Value'
    END AS customer_segment
FROM ranked_customers
ORDER BY total_revenue_12m DESC;

StatementMeta(, 4a62ac9d-16fb-413d-8a0f-b8623e82b568, 7, Finished, Available, Finished, False)

<Spark SQL result set with 16 rows and 4 fields>

In [ ]:
%%sql
WITH product_pairs AS (
    SELECT
        f1.order_id,
        f1.product_id AS product_id_1,
        f2.product_id AS product_id_2
    FROM fact_sales f1
    JOIN fact_sales f2
        ON f1.order_id = f2.order_id
       AND f1.product_id < f2.product_id
),
pair_counts AS (
    SELECT
        pp.product_id_1,
        p1.product_title AS product_1,
        pp.product_id_2,
        p2.product_title AS product_2,
        COUNT(*) AS joint_frequency
    FROM product_pairs pp
    JOIN dim_product p1
        ON pp.product_id_1 = p1.product_id
    JOIN dim_product p2
        ON pp.product_id_2 = p2.product_id
    GROUP BY
        pp.product_id_1,
        p1.product_title,
        pp.product_id_2,
        p2.product_title
)
SELECT
    product_1,
    product_2,
    joint_frequency
FROM pair_counts
ORDER BY joint_frequency DESC, product_1, product_2
LIMIT 10;

StatementMeta(, 4a62ac9d-16fb-413d-8a0f-b8623e82b568, 6, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

In [ ]:
%%sql
WITH customer_quarters AS (
    SELECT
        f.customer_id,
        COUNT(DISTINCT CONCAT(CAST(d.year AS STRING), '-Q', CAST(d.quarter AS STRING))) AS quarter_count
    FROM fact_sales f
    JOIN dim_date d
        ON f.date_key = d.date_key
    GROUP BY f.customer_id
),
customer_stats AS (
    SELECT
        COUNT(*) AS total_customers,
        SUM(CASE WHEN quarter_count > 1 THEN 1 ELSE 0 END) AS customers_multi_quarter
    FROM customer_quarters
)
SELECT
    total_customers,
    customers_multi_quarter,
    ROUND(100.0 * customers_multi_quarter / total_customers, 2) AS percentage_multi_quarter
FROM customer_stats;

StatementMeta(, 4a62ac9d-16fb-413d-8a0f-b8623e82b568, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>